# Production RAG operations: release and recover safely

## Northstar release scenario

A new reranker improves a demo but may increase p95 latency and cost. Meanwhile, one tenant’s index is stale. This notebook turns those conditions into observable release, readiness, degradation, and rollback decisions.


## Operational control loop

```text
[Versioned change] -> [Offline evaluation] -> [Readiness + freshness] -> [Release gate]
                                                        | pass
                                                        v
[Telemetry] <- [Canary] <- [Promote] -> [Monitor quality, safety, latency, cost]
    | regression
    v
[Contain: circuit breaker / kill switch / rollback] -> [Verify] -> [Post-incident regression]
```

An operationally successful RAG system can explain both its answer and its current operating condition.


### Render the Mermaid diagram in Jupyter

Run the next cell in JupyterLab or a compatible notebook viewer to render the Mermaid operational lifecycle. GitHub reliably renders Mermaid in the companion Markdown lesson; this cell makes the notebook visualization explicit for local learners.


In [ ]:
from IPython.display import HTML, display

mermaid_source = """
flowchart TD
  R[Request] --> T[Trace route, retrieval, generation]
  T --> B{Latency / cost / safety budget}
  B -->|pass| V[Answer verification]
  B -->|exceeded| F[Degrade, shed load, or abstain]
  I[Index + corpus freshness] --> H[Readiness]
  E[Golden-set / online evaluation] --> G[Release gate]
  H --> G
  G -->|promote| C[Canary + monitor]
  G -->|hold / regression| RB[Rollback or kill switch]
"""

display(HTML(f"""
<script src="https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js"></script>
<div class="mermaid">{mermaid_source}</div>
<script>mermaid.initialize({{startOnLoad:true, theme:'base'}});</script>
"""))


In [ ]:
from datetime import datetime, timedelta, timezone
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/advanced/05-production-operations/lab.py')).items() if not name.startswith('_')})

now = datetime.now(timezone.utc)
trace = Trace("How do I rotate a key?", "corrective-retrieval", trace_id="trace-2026-08-01")
trace.record("retrieval:start")
trace.record("retrieval:3-authorized-candidates")
trace.record("generation:verified-citations")
trace.latency_ms, trace.cost_usd = 1450, 0.012
print(trace)


## 1 — Budget each request trajectory

Budgets are controls, not dashboards. Apply them before expensive recovery or generation steps, then choose an explicit degraded route or abstention if a budget is exceeded.


In [ ]:
budget = Budget(max_latency_ms=2500, max_cost_usd=0.02)
print("within budget:", within_budget(trace, budget))
slow = Trace("q", "agentic", latency_ms=4200, cost_usd=0.01)
expensive = Trace("q", "agentic", latency_ms=800, cost_usd=0.08)
assert not within_budget(slow, budget)
assert not within_budget(expensive, budget)
print("safe fallback for slow route: skip alternate reranker and return supported baseline or abstain")


## 2 — Readiness is not freshness

A healthy API can still serve a stale corpus. Evaluate these independently and expose both to a deployment gate and request router.


In [ ]:
fresh = freshness_status(now - timedelta(hours=2), now, max_age_hours=12)
stale = freshness_status(now - timedelta(hours=30), now, max_age_hours=12)
ready = health_status(index_ready=True, evaluator_ready=True, corpus_fresh=fresh == "fresh")
stale_health = health_status(index_ready=True, evaluator_ready=True, corpus_fresh=stale == "fresh")
print(ready)
print(stale_health)
assert ready == {"ready": "ok", "corpus": "fresh"}
assert stale_health["corpus"] == "stale"


## 3 — Gate a release with evidence

A release requires quality, dependency readiness, corpus freshness, and error-rate checks. A feature flag or canary is the *next* step after a gate, not a substitute for it.


In [ ]:
promote = release_gate(quality_score=0.91, min_quality=0.88, readiness=ready, error_rate=0.004, max_error_rate=0.01)
hold = release_gate(quality_score=0.86, min_quality=0.88, readiness=stale_health, error_rate=0.03, max_error_rate=0.01)
print(promote)
print(hold)
assert promote["decision"] == "promote"
assert hold["decision"] == "hold"
assert "golden-set-quality-below-threshold" in hold["reasons"]


## 4 — Canary analysis: compare a candidate with the baseline

Do not compare only model calls. Compare supported-answer rate, citation verification, p95 latency, cost per supported answer, and safety events. A candidate must not trade away a hard safety or freshness constraint for a small relevance gain.


In [ ]:
baseline = {"supported_rate": .90, "p95_ms": 1800, "cost_per_supported": .018, "citation_failures": 0}
candidate = {"supported_rate": .92, "p95_ms": 3400, "cost_per_supported": .041, "citation_failures": 1}
def canary_decision(baseline, candidate):
    if candidate["citation_failures"]: return "rollback: citation verification failure"
    if candidate["p95_ms"] > 3000: return "hold: latency SLO violated"
    if candidate["cost_per_supported"] > baseline["cost_per_supported"] * 1.5: return "hold: cost regression"
    return "continue-canary"
print(canary_decision(baseline, candidate))
assert canary_decision(baseline, candidate).startswith("rollback")


## 5 — Circuit breakers and safe degradation

A broken verifier, vector store, or external-search route should not cause unlimited retries. Circuit breaking moves the system to a documented degraded mode: cache only identity-safe evidence, use an approved baseline, or abstain.


In [ ]:
for failures in range(5):
    print(failures, circuit_breaker(consecutive_failures=failures, threshold=3))
assert circuit_breaker(consecutive_failures=3, threshold=3) == "open"
def degraded_route(index_ready, verifier_ready):
    if not verifier_ready: return "abstain: answer verification unavailable"
    if not index_ready: return "fallback: approved cached evidence only"
    return "normal"
print(degraded_route(index_ready=False, verifier_ready=True))
assert degraded_route(False, True).startswith("fallback")


## 6 — Trace hygiene and incident response

Keep identifiers, versions, timings, route decisions, and candidate IDs. Redact raw user text, secrets, private documents, and tool payloads unless a narrowly controlled incident process requires them. Every alert needs an owner, threshold rationale, runbook, and containment action.


In [ ]:
safe_trace = {
    "trace_id": trace.trace_id, "tenant_hash": "t:8a9f", "route": trace.route,
    "events": trace.events, "latency_ms": trace.latency_ms, "cost_usd": trace.cost_usd,
    "index_version": "2026-08-01", "reranker_version": "candidate-b",
}
print(safe_trace)
assert "How do I rotate" not in repr(safe_trace)


## 7 — Production evaluation record

A release record combines offline and online evidence. This compact gate is intentionally deterministic; production teams should retain the raw metric distributions, dataset versions, statistical confidence, owners, and rollback links.


In [ ]:
record = {
    "quality": 0.91, "readiness": ready, "error_rate": 0.004,
    "no_cross_tenant_leak": True, "freshness": fresh, "rollback_version": "reranker-a",
}
record["releaseable"] = (release_gate(quality_score=record["quality"], min_quality=.88, readiness=record["readiness"], error_rate=record["error_rate"], max_error_rate=.01)["decision"] == "promote" and record["no_cross_tenant_leak"])
print(record)
assert record["releaseable"]


## 8 — Attribute latency to pipeline stages

A single total hides the remediation target. Break down retrieval, reranking, generation, verification, and external dependencies. Alert on p95/p99 by stage, not only average request time.


In [ ]:
stage_ms = {"auth_filter": 18, "retrieve": 340, "rerank": 710, "generate": 980, "verify": 210}
total_ms = sum(stage_ms.values())
dominant = max(stage_ms, key=stage_ms.get)
print({"total_ms": total_ms, "dominant_stage": dominant, "share": round(stage_ms[dominant] / total_ms, 2)})
assert dominant == "generate"
assert total_ms == 2258


## 9 — Tenant-specific freshness and routing

A global “index healthy” signal is often insufficient. A tenant may have a different ingestion cadence, data residency requirement, or freshness SLO. Route stale tenants to a dated response, an approved baseline, or abstention—never silently claim current knowledge.


In [ ]:
tenant_freshness_hours = {"northstar": 2, "regulated-acme": 30}
tenant_slo_hours = {"northstar": 24, "regulated-acme": 12}
def tenant_route(tenant):
    if tenant_freshness_hours[tenant] > tenant_slo_hours[tenant]:
        return "stale: disclose date and escalate for refresh"
    return "normal-rag"
print({tenant: tenant_route(tenant) for tenant in tenant_freshness_hours})
assert tenant_route("regulated-acme").startswith("stale")


## 10 — Alert classification and runbook ownership

High-signal alerts must name an owner and a containment action. This small router is a template for a real alert policy, not a replacement for paging, dashboards, or incident management.


In [ ]:
def classify_alert(signal: str) -> dict[str, str]:
    table = {
        "citation_failure": {"severity": "high", "owner": "rag-quality", "containment": "disable answer route; require abstention"},
        "stale_index": {"severity": "medium", "owner": "data-platform", "containment": "serve dated results; trigger reindex"},
        "cross_tenant_denial": {"severity": "critical", "owner": "security", "containment": "block request; investigate policy trace"},
    }
    return table[signal]
print(classify_alert("citation_failure"))
assert classify_alert("cross_tenant_denial")["severity"] == "critical"


## 11 — Canary progression and rollback compatibility

A rollback is only safe when the prior index, prompt, embedding, cache, and schema versions remain compatible. Record a rollout plan before changing traffic; do not discover rollback dependencies during an incident.


In [ ]:
rollout = [
    {"traffic": 1, "quality": .91, "p95_ms": 2100, "citation_failures": 0},
    {"traffic": 10, "quality": .92, "p95_ms": 2700, "citation_failures": 0},
    {"traffic": 25, "quality": .92, "p95_ms": 3200, "citation_failures": 0},
]
def rollout_action(point):
    if point["citation_failures"] or point["p95_ms"] > 3000: return "rollback-to-reranker-a"
    return "advance-canary"
print([(p["traffic"], rollout_action(p)) for p in rollout])
assert rollout_action(rollout[-1]) == "rollback-to-reranker-a"


## 12 — Verify degraded mode preserves hard controls

A degradation path may reduce recall or availability, but it must not disable tenant filtering, citation verification, or approval gates. Test these invariants like any other safety requirement.


In [ ]:
degraded = {"external_search": False, "reranker": "baseline", "tenant_filter": True, "citation_verifier": True, "actions_enabled": False}
assert degraded["tenant_filter"] and degraded["citation_verifier"]
assert not degraded["actions_enabled"]
print("Safe degraded mode:", degraded)


## Exercises

1. Add stage-level retrieval, rerank, and generation timing and identify the largest p95 contributor.
2. Simulate a tenant-specific stale index and design the visible user response.
3. Add a release gate for prompt-injection and cross-tenant regression cases.
4. Build a rollback plan for an embedding-model upgrade with index/cache compatibility checks.
5. Add a cost circuit breaker and prove it cannot disable citation verification.
6. Draft an incident runbook for a spike in empty retrievals, including dashboard, owner, containment, and postmortem test.

See the companion [lesson](README.md) for SLO, governance, technology, and operational guidance.
